# 03 · LMGECplus — the modular framework

`LMGECplus` is a thin facade over interchangeable strategy objects. Every algorithmic ingredient — reconstruction loss, normalization, graph propagation, view weighting / attention, cross-view alignment, clustering head, optimizer — is a component you can swap by name, by dict, or by passing your own instance.

With **no overrides it reproduces the LMGEC baseline exactly**, which is what makes clean ablations possible.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow

import numpy as np
import scipy.sparse as sp


def make_synthetic(n_per=60, k=3, dims=(30, 20), sep=4.0, seed=0):
    """A small, well-separated multi-view dataset: (Xs, As, labels).

    Each view is Gaussian blobs around k cluster centres, plus a sparse
    intra-cluster adjacency so graph propagation has something to smooth.
    """
    rng = np.random.RandomState(seed)
    n = n_per * k
    labels = np.repeat(np.arange(k), n_per)
    Xs, As = [], []
    for d in dims:
        centres = rng.randn(k, d) * sep
        X = centres[labels] + rng.randn(n, d)
        Xs.append(X.astype(np.float64))
        A = np.zeros((n, n))
        for i in range(n):
            same = np.where(labels == labels[i])[0]
            nbrs = rng.choice(same, size=min(6, len(same)), replace=False)
            A[i, nbrs] = 1.0
        As.append(sp.csr_matrix(np.maximum(A, A.T)))
    return Xs, As, labels


Xs, As, labels = make_synthetic()

# --- To use a real benchmark instead (downloads + caches on first call): ---
# from pymv.data import load_dataset
# data = load_dataset("acm")
# Xs, As, labels = data.Xs, data.As, data.labels

k = len(np.unique(labels))
print(f"views={len(Xs)}  samples={Xs[0].shape[0]}  clusters={k}")
print("view feature dims:", [X.shape[1] for X in Xs])

## What's on the shelf

In [ ]:
from pymv.models.lmgecplus import list_components

for kind, names in list_components().items():
    print(f'{kind:14s}: {names}')

## Baseline parity check

LMGECplus with defaults should match LMGEC's fused weights and clustering.

In [ ]:
from pymv.models import LMGEC, LMGECplus
from pymv.metrics import clustering_accuracy

base = LMGEC(n_clusters=k, embedding_dim=16, max_iter=30).fit(Xs, As)
plus = LMGECplus(n_clusters=k, embedding_dim=16, max_iter=30).fit(Xs, As)
print('LMGEC     alphas:', np.round(base.alphas_, 3), 'ACC', round(clustering_accuracy(labels, base.labels_), 3))
print('LMGECplus alphas:', np.round(plus.alphas_, 3), 'ACC', round(clustering_accuracy(labels, plus.labels_), 3))

## Swap one component at a time

Three equivalent spec styles: a **string** (`"l21"`), a **dict** with constructor kwargs (`{"name": "static", "dim_normalized": True}`), or a component **instance**.

In [ ]:
variants = {
    'baseline'         : {},
    'robust l21 loss'  : {'reconstruction_loss': 'l21'},
    'huber loss'       : {'reconstruction_loss': 'huber'},
    'node attention'   : {'attention': 'node'},
    'view attention'   : {'attention': 'view'},
    'soft clustering'  : {'clustering': 'soft'},
    'gmm clustering'   : {'clustering': 'gmm'},
    'controlled prop'  : {'graph_propagation': {'enabled': True, 'gamma': 0.5}},
    'cosine alignment' : {'alignment': 'cosine'},
    'zscore norm'      : {'normalization': 'zscore'},
}

for name, kw in variants.items():
    m = LMGECplus(n_clusters=k, embedding_dim=16, max_iter=30, **kw).fit(Xs, As)
    print(f'{name:18s}  ACC={clustering_accuracy(labels, m.labels_):.3f}')

## The structured result object

Every `LMGECplus.fit` populates `results_` with all intermediates — handy for analysing an ablation.

In [ ]:
m = LMGECplus(n_clusters=k, embedding_dim=16, attention='node', max_iter=30).fit(Xs, As)
r = m.results_
print('components   :', r.components)
print('n iterations :', r.convergence.get('n_iter'))
print('final loss   :', round(r.convergence.get('final_loss', float('nan')), 4))
print('train time   :', round(r.training_time, 4), 's')
print('attention_   :', None if m.attention_ is None else m.attention_.shape)

## Add your own component in one class

`@register_component(kind, name)` plugs a new strategy into the framework — no library edits, immediately usable by name.

In [ ]:
from pymv.models.lmgecplus import ReconstructionLoss, register_component
import numpy as np

@register_component('loss', 'cauchy')
class CauchyLoss(ReconstructionLoss):
    """A robust Cauchy penalty as a drop-in reconstruction loss."""
    def __init__(self, c=1.0):
        self.c = c
    def penalty(self, r):
        return np.log1p((r / self.c) ** 2).sum()
    def irls_node_weights(self, r):
        return 1.0 / (1.0 + (r / self.c) ** 2)

m = LMGECplus(n_clusters=k, embedding_dim=16, reconstruction_loss='cauchy', max_iter=30).fit(Xs, As)
print('cauchy loss  ACC =', round(clustering_accuracy(labels, m.labels_), 3))